# Chapter 11 — Review and customize a schematic

TARGET API · CONVERGING · not executable on the current runtime

> **TARGET API / CONVERGING — not executable on the current runtime.**
> Layout references live authored handles and does not create
> independent topology.

Two coupled grounded LC subsystems can be electrically clear yet
visually hard to read. This Chapter reviews the complete literal-value
circuit first, then uses pure presentation hints without changing
physical flow or selecting an analysis View.

## Lesson 11.1 — Author two coupled LC subsystems

### Declare the readout LC subsystem

Start with the root and its readout child so the child owns its native
leaves, parallel relation, and one public `readout_terminal` boundary.

In [ ]:
from pathlib import Path

from scnsim import (
    CircuitDiagramSpec,
    CircuitPlan,
    DiagramAxis,
    DiagramSide,
    SchematicLayout,
    Theme,
    components,
    units as u,
)

plan = CircuitPlan(id="review_structured_schematic")
readout = plan.subsystem(id="readout")
readout_capacitor = readout.add(
    components.capacitor(
        id="capacitor",
        capacitance=110.0 * u.fF,
    )
)
readout_inductor = readout.add(
    components.inductor(
        id="inductor",
        inductance=5.8 * u.nH,
    )
)
readout_terminal_bus = readout.bus(id="terminal")
readout_parallel = readout.parallel(
    id="parallel_lc",
    start=readout_terminal_bus,
    branches=((readout_capacitor,), (readout_inductor,)),
    end=readout.ground,
)
readout_terminal = readout.expose_pin(
    id="terminal",
    at=readout_terminal_bus,
)

The returned terminal and `readout_parallel` handle are the only readout
references used by root wiring and later presentation hints; the LC
values are literal native component values in this lesson.

### Declare the storage LC subsystem

Repeat the independent child declaration for storage; it publishes its
own terminal rather than sharing readout internals.

In [ ]:
storage = plan.subsystem(id="storage")
storage_capacitor = storage.add(
    components.capacitor(
        id="capacitor",
        capacitance=100.0 * u.fF,
    )
)
storage_inductor = storage.add(
    components.inductor(
        id="inductor",
        inductance=6.0 * u.nH,
    )
)
storage_terminal_bus = storage.bus(id="terminal")
storage_parallel = storage.parallel(
    id="parallel_lc",
    start=storage_terminal_bus,
    branches=((storage_capacitor,), (storage_inductor,)),
    end=storage.ground,
)
storage_terminal = storage.expose_pin(
    id="terminal",
    at=storage_terminal_bus,
)

`storage_terminal` and `storage_parallel` now provide the root boundary
and the branch handle consumed by the layout cell.

### Add the root coupling

The 6 fF element stays between separate root buses; the child terminals
are linked only to their respective root nodes.

In [ ]:
readout_node_bus = plan.bus(id="readout_node")
storage_node_bus = plan.bus(id="storage_node")
coupler = plan.add(
    components.capacitor(
        id="readout_storage_coupler",
        capacitance=6.0 * u.fF,
    )
)
coupling = plan.series(
    id="readout_storage",
    start=readout_node_bus,
    elements=(coupler,),
    end=storage_node_bus,
)
plan.link(
    id="readout_child",
    endpoints=(readout_node_bus, readout_terminal),
)
plan.link(
    id="storage_child",
    endpoints=(storage_node_bus, storage_terminal),
)

The root has two separate nodes, the ordered `coupling` relation, and
one link per public child terminal; the next cell promotes their
terminated boundaries.

### Promote the two terminated Ports

In [ ]:
readout_port = plan.add_port(
    id="readout_port",
    at=readout_node_bus,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)
storage_port = plan.add_port(
    id="storage_port",
    at=storage_node_bus,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)

`readout` and `storage` are complete inline child regions and
independent root peers: their native leaves are already assembled inside
their own parallel relations. The root’s 6 fF coupler is a separate
structural use, while the direct links bind public boundaries without
adding component slots. The Ports complete the physical declaration and
remain root-bus handles for automatic and hinted presentation.

## Lesson 11.2 — Compare automatic and hinted presentation

### Render the automatic layout

In [ ]:
automatic = plan.render_schematic(
    CircuitDiagramSpec(
        representation="authoring",
        theme=Theme.AUTO,
        show_parameter_values=True,
        show_provenance=True,
        layout=None,
    )
)
automatic.show()

`automatic` records the declaration-order presentation without a layout
hint; keep it to compare with the next, explicitly hinted rendering.

### Construct explicit presentation hints

Scope `order` names only direct peer subsystems and must list every
eligible peer exactly once; omitting a peer is a layout validation
failure, not a partial hint. ParallelRef keys may permute only their
complete `.branches` tuples; electrical series order is immutable.

In [ ]:
layout = SchematicLayout(
    axes={
        readout: DiagramAxis.VERTICAL,
        storage: DiagramAxis.VERTICAL,
    },
    order={
        plan: (readout, storage),
        readout_parallel: tuple(reversed(readout_parallel.branches)),
        storage_parallel: storage_parallel.branches,
    },
    terminal_sides={
        readout_terminal: DiagramSide.RIGHT,
        storage_terminal: DiagramSide.LEFT,
    },
    port_sides={
        readout_port: DiagramSide.LEFT,
        storage_port: DiagramSide.RIGHT,
    },
)

`layout` changes only axes, peer order, branch order, and terminal/Port
sides; it consumes authored handles without creating topology.

### Render the hinted layout

In [ ]:
hinted = plan.render_schematic(
    CircuitDiagramSpec(
        representation="authoring",
        theme=Theme.AUTO,
        show_parameter_values=True,
        show_provenance=True,
        layout=layout,
    )
)
hinted_drawing = hinted.show()

`hinted` is a second presentation of the same Plan. Review its audit
alongside the automatic drawing before comparing their declaration
identities. Prepare the relative export directory explicitly in a fresh
kernel before saving it.

### Export the hinted drawing

In [ ]:
export_path = Path(
    "workspaces/advanced-course/review_structured_schematic.svg"
)
export_path.parent.mkdir(parents=True, exist_ok=True)
hinted_drawing.save(export_path)

`export_path` names a target-only artifact location. The next cells
remain the separate audit and declaration-identity reviews.

### Inspect the drawings’ audit tables

In [ ]:
automatic.audit.show()

In [ ]:
hinted.audit.show()

Each audit table is visible independently; the next cell checks the
three declaration digest classes directly.

### Compare declaration identities

In [ ]:
assert automatic.audit.plan_sha256 == hinted.audit.plan_sha256
assert automatic.audit.connectivity_sha256 == hinted.audit.connectivity_sha256
assert automatic.audit.semantic_sha256 == hinted.audit.semantic_sha256

Matching digests confirm that the layout hints did not alter the Plan,
connectivity, or semantic declaration.

Hints alter only presentation. By default, scope peers and series run
horizontally; each parallel branch runs horizontally from start to end,
and the branches stack top-to-bottom in declared order.
`DiagramAxis.VERTICAL` rotates that local group, so branches run
top-to-bottom and stack left-to-right. It is not inferred from a main
bus, weighted diameter, or physical flow.

## Lesson 11.3 — Stitch two public terminals

### Stitch public pins in a separate Plan

This separate zero-component conductive stitch contains two real
grounded LC children. Its one shared root bus joins their public
boundaries directly, so it does not short the main Plan’s 6 fF coupling
element.

In [ ]:
stitch_plan = CircuitPlan(id="direct_stitch")
left = stitch_plan.subsystem(id="left")
left_capacitor = left.add(
    components.capacitor(id="capacitor", capacitance=110.0 * u.fF)
)
left_inductor = left.add(
    components.inductor(id="inductor", inductance=5.8 * u.nH)
)
left_terminal_bus = left.bus(id="terminal")
left_parallel = left.parallel(
    id="parallel_lc",
    start=left_terminal_bus,
    branches=((left_capacitor,), (left_inductor,)),
    end=left.ground,
)
left_terminal = left.expose_pin(id="terminal", at=left_terminal_bus)

The left child is nonempty and publishes `left_terminal`; the next cell
builds the equally independent right child for the shared-node stitch.

In [ ]:
right = stitch_plan.subsystem(id="right")
right_capacitor = right.add(
    components.capacitor(id="capacitor", capacitance=100.0 * u.fF)
)
right_inductor = right.add(
    components.inductor(id="inductor", inductance=6.0 * u.nH)
)
right_terminal_bus = right.bus(id="terminal")
right_parallel = right.parallel(
    id="parallel_lc",
    start=right_terminal_bus,
    branches=((right_capacitor,), (right_inductor,)),
    end=right.ground,
)
right_terminal = right.expose_pin(id="terminal", at=right_terminal_bus)

Both public terminals are ready for one N-endpoint direct link to the
root bus.

In [ ]:
stitch_bus = stitch_plan.bus(id="stitched_node")
stitch_plan.link(
    id="children_stitch",
    endpoints=(stitch_bus, left_terminal, right_terminal),
)

`stitch_bus` is one shared electrical net, so this Plan teaches a
zero-component stitch without touching the main coupler.

## Lesson 11.4 — Ground a public return at its parent

### Make the public return boundary explicit

An ordinary child bus named `return` is not intrinsically ground. This
complete module publishes it, and the parent deliberately grounds the
public return pin; the grounding belongs to the assembly rather than
changing the child’s schema.

In [ ]:
return_plan = CircuitPlan(id="external_return")
return_module = return_plan.subsystem(id="module")
module_signal = return_module.bus(id="signal")
module_return = return_module.bus(id="return")
module_capacitor = return_module.add(
    components.capacitor(id="capacitor", capacitance=6.0 * u.fF)
)
return_module.series(
    id="capacitor_path",
    start=module_signal,
    elements=(module_capacitor,),
    end=module_return,
)
module_signal_pin = return_module.expose_pin(id="signal", at=module_signal)
module_return_pin = return_module.expose_pin(id="return", at=module_return)
return_plan.ground_pins(pins=(module_return_pin,))

`ground_pins` grounds the whole currently connected return net, not a
drawing marker. It accepts public pins at this boundary; a parent must
never use the child-private `module_return` bus directly. A raw
`GroundRef` is not a `link` endpoint, and a Port’s own load-to-ground
policy needs no public ground pin.

## Lesson 11.5 — Common independent returns

### Ground distinct returns without inventing a rail

Distinct public return pins may be grounded in one call when their
modules are otherwise independent. This is ordinary common reference,
not an implicit short between the modules’ signal terminals or a new
graphical grouping.

In [ ]:
two_return_plan = CircuitPlan(id="two_public_returns")
left_return = two_return_plan.subsystem(id="left")
right_return = two_return_plan.subsystem(id="right")
left_signal = left_return.bus(id="signal")
left_return_bus = left_return.bus(id="return")
right_signal = right_return.bus(id="signal")
right_return_bus = right_return.bus(id="return")
left_capacitor = left_return.add(
    components.capacitor(id="capacitor", capacitance=6.0 * u.fF)
)
right_capacitor = right_return.add(
    components.capacitor(id="capacitor", capacitance=6.0 * u.fF)
)
left_return.series(
    id="capacitor_path",
    start=left_signal,
    elements=(left_capacitor,),
    end=left_return_bus,
)
right_return.series(
    id="capacitor_path",
    start=right_signal,
    elements=(right_capacitor,),
    end=right_return_bus,
)
left_return_pin = left_return.expose_pin(id="return", at=left_return_bus)
right_return_pin = right_return.expose_pin(id="return", at=right_return_bus)
two_return_plan.ground_pins(pins=(left_return_pin, right_return_pin))

Normal return commoning grounds the two return nets only. Connecting
both ends of either capacitor to one net would instead short that
physical element and is rejected by authoring validation.

## Lesson 11.6 — Correct expected authoring validation

### Read each safety diagnostic before correcting the declaration

The following target-only cases are complete small declarations. They
keep the failed mutation visible, preserve its reusable structure id,
and then use a legal public boundary or corrected physical relation. No
cell assumes the current runtime executes these target validations.

In [ ]:
from scnsim import SCNSimValidationError

parent_signal = return_plan.bus(id="parent_signal")
try:
    return_plan.link(
        id="return_attachment",
        endpoints=(parent_signal, module_signal),
    )
except SCNSimValidationError as private_bus_error:
    print(private_bus_error)
    return_plan.link(
        id="return_attachment",
        endpoints=(parent_signal, module_signal_pin),
    )

`module_signal` remains child-private even though this Chapter still has
its Python variable. The retry reuses the rejected id only after
replacing the private bus with the immediate child’s public signal pin.

In [ ]:
conflict_plan = CircuitPlan(id="transitive_bus_conflict")
conflict_child = conflict_plan.subsystem(id="child")
conflict_capacitor = conflict_child.add(
    components.capacitor(id="capacitor", capacitance=110.0 * u.fF)
)
conflict_inductor = conflict_child.add(
    components.inductor(id="inductor", inductance=5.8 * u.nH)
)
conflict_internal = conflict_child.bus(id="terminal")
conflict_child.parallel(
    id="parallel_lc",
    start=conflict_internal,
    branches=((conflict_capacitor,), (conflict_inductor,)),
    end=conflict_child.ground,
)
conflict_pin = conflict_child.expose_pin(id="terminal", at=conflict_internal)
first_bus = conflict_plan.bus(id="first")
second_bus = conflict_plan.bus(id="second")
conflict_coupler = conflict_plan.add(
    components.capacitor(id="coupler", capacitance=6.0 * u.fF)
)
conflict_plan.link(id="first_attachment", endpoints=(first_bus, conflict_pin))
try:
    conflict_plan.link(
        id="second_attachment",
        endpoints=(second_bus, conflict_pin),
    )
except SCNSimValidationError as second_bus_error:
    print(second_bus_error)
    conflict_plan.series(
        id="second_attachment",
        start=second_bus,
        elements=(conflict_coupler,),
        end=first_bus,
    )

The second call would transitively merge two distinct parent buses, so
it is rejected rather than silently creating an anonymous parent short.

In [ ]:
invalid_plan = CircuitPlan(id="invalid_connections")
shorted_bus = invalid_plan.bus(id="shorted")
shorted_capacitor = invalid_plan.add(
    components.capacitor(id="capacitor", capacitance=6.0 * u.fF)
)
try:
    invalid_plan.series(
        id="shorted_capacitor",
        start=shorted_bus,
        elements=(shorted_capacitor,),
        end=shorted_bus,
    )
except SCNSimValidationError as shorted_element_error:
    print(shorted_element_error)
    distinct_bus = invalid_plan.bus(id="distinct")
    invalid_plan.series(
        id="shorted_capacitor",
        start=shorted_bus,
        elements=(shorted_capacitor,),
        end=distinct_bus,
    )

The repaired relation uses the same capacitor between distinct buses.
The next complete Plan isolates the separate rule for a Port-bound
signal net.

In [ ]:
grounded_port_plan = CircuitPlan(id="grounded_port")
port_child = grounded_port_plan.subsystem(id="module")
port_child_signal = port_child.bus(id="signal")
port_child_return = port_child.bus(id="return")
port_child_capacitor = port_child.add(
    components.capacitor(id="capacitor", capacitance=6.0 * u.fF)
)
port_child.series(
    id="port_capacitor",
    start=port_child_signal,
    elements=(port_child_capacitor,),
    end=port_child_return,
)
port_signal_pin = port_child.expose_pin(id="signal", at=port_child_signal)
port_return_pin = port_child.expose_pin(id="return", at=port_child_return)
port_bus = grounded_port_plan.bus(id="port")
grounded_port_plan.link(
    id="port_signal_attachment",
    endpoints=(port_bus, port_signal_pin),
)
grounded_port_plan.add_port(
    id="terminated",
    at=port_bus,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)
try:
    grounded_port_plan.ground_pins(pins=(port_signal_pin,))
except SCNSimValidationError as grounded_port_error:
    print(grounded_port_error)
grounded_port_plan.ground_pins(pins=(port_return_pin,))

A Port already carries its declared load-to-ground policy and is not a
public ground pin. `port_signal_pin` is an eligible immediate-child
public `PinRef` on the Port-bound final net, so its rejection
demonstrates the grounded-Port authoring invariant. The other capacitor
terminal remains distinct and ungrounded before that call, so the
earlier rejection is separately the shorted-element case. The legal
correction grounds the return pin, not the signal pin carrying the Port.

In [ ]:
return_diagram = return_plan.render_schematic(CircuitDiagramSpec())
return_diagram.show()

In [ ]:
return_diagram.audit.show()

[Previous](10_use_custom_component.qmd) ·
[Next](12_model_n_trace_line.qmd)